# C389 CA1: tuning runs (4 classes, Twist 2)

Fine-tunes our app model **v4 26m** (`final_4cls_ig_v4_m`) on the same `final_ig_v4.zip` data: person_left, person_right, wheeled_left, wheeled_right (the lecturer's Twist 2 classes). A rider and their bike are one wheeled box.

**Tuning round (8-9 Nov).** A check on the validation set showed v4 misses more people who are half hidden behind someone (67% found, against 89% for people in the clear), misses small far-away people (59% found), and gives some people the wrong direction. Four runs each try one fix (Settings, `EXPERIMENT` = A, B, C or D), so team members can run them at the same time on their own Colab accounts.

- The zip also holds the **test** split. This notebook never reads it (section 4 removes it from data.yaml). The runs are compared on **validation** counting accuracy only.
- Before running: **Runtime → Change runtime type → T4 GPU**.
- Using your own Google account: the owner shares the `C389_Supernova_Model` folder with you, and you add it to My Drive (right-click → Organise → Add shortcut), so the paths below work unchanged.


## 1. Settings

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/C389_Supernova_Model"
ZIP_PATH   = f"{DRIVE_ROOT}/data/final_ig_v4.zip"   # the v4 data the app model was trained on (no mirrors)
RUNS_DIR   = f"{DRIVE_ROOT}/runs"

DATA_DIR   = "/content/final_ig_v4"   # unzipped on Colab's local disk (faster than Drive)

# Tuning round (8-9 Nov): every run fine-tunes the app model v4 26m on the same v4 data and changes one
# idea, so the runs can go in parallel on different Colab accounts. Pick ONE letter per person:
#   "A" bigger input (800): small and half-hidden people get more pixels.       about 2.5 h on a T4
#   "B" direction: classification loss x2 and the last 15 epochs without mosaic,
#       aimed at people found but given the wrong direction.                     about 1.5 h
#   "C" crowds: more zoom-out and mixup, so the model sees more small and
#       overlapping people.                                                      about 1.5 h
#   "D" polish: lower learning rate, 60 epochs, cosine schedule.               about 2 h
# The runs are compared on VALID counting accuracy only. Test is not used to choose.
EXPERIMENT = "A"

V4_WEIGHTS = f"{RUNS_DIR}/final_4cls_ig_v4_m/weights/best.pt"
MODEL_VARIANT = V4_WEIGHTS

# Shared by all four: fine-tuning settings (AdamW with a small fixed learning rate, because the
# optimizer="auto" default ignores lr0) and the v4 data.
COMMON = dict(optimizer="AdamW", lr0=0.0005, epochs=40, imgsz=640, batch=16)
EXPERIMENTS = {
    "A": dict(imgsz=800, batch=8),
    "B": dict(cls=1.0, close_mosaic=15),
    "C": dict(scale=0.7, mixup=0.1, close_mosaic=10),
    "D": dict(lr0=0.0002, epochs=60, cos_lr=True),
}
TRAIN_ARGS = {**COMMON, **EXPERIMENTS[EXPERIMENT]}
RUN_NAME   = f"tune_v4_{EXPERIMENT}"
EPOCHS     = TRAIN_ARGS["epochs"]
IMAGE_SIZE = TRAIN_ARGS["imgsz"]
BATCH_SIZE = TRAIN_ARGS["batch"]
print(RUN_NAME, TRAIN_ARGS)

# Colab disconnected or out of time? Set True and run sections 1-5 again (even on another day):
# it carries on from last.pt on Drive, which is saved after every epoch.
RESUME = False

# Not a choice: Ultralytics' random flip keeps the label, so a flipped person_left would still say left
# while moving right.
FLIP_LEFT_RIGHT = 0.0

EXPECTED_CLASSES = ["person_left", "person_right", "wheeled_left", "wheeled_right"]


## 2. Install and check the GPU

In [ ]:
!pip install -q ultralytics==8.4.120

import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU. Runtime -> Change runtime type -> T4 GPU, then run again.")

## 3. Connect Drive and unzip the dataset

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(f"Zip not found at {ZIP_PATH}. Check the folder name in DRIVE_ROOT.")

!unzip -q -o "{ZIP_PATH}" -d /content/
os.makedirs(RUNS_DIR, exist_ok=True)

for split in ("train", "valid", "test"):
    folder = os.path.join(DATA_DIR, split, "images")
    n = len(os.listdir(folder)) if os.path.isdir(folder) else 0
    print(f"{split:6} {n:4d} images")

## 4. Check the classes and fix the dataset path

In [ ]:
import yaml

yaml_path = os.path.join(DATA_DIR, "data.yaml")
cfg = yaml.safe_load(open(yaml_path))
names = list(cfg["names"].values()) if isinstance(cfg["names"], dict) else list(cfg["names"])
print("classes:", names)
if names != EXPECTED_CLASSES:
    raise ValueError(f"Classes are {names}, expected {EXPECTED_CLASSES}. Run remap_classes on the export first.")

cfg["path"] = DATA_DIR
cfg.pop("test", None)  # never train or tune on test
yaml.safe_dump(cfg, open(yaml_path, "w"), sort_keys=False)
print(open(yaml_path).read())

## 5. Train

About 1.7 hours on a T4 when fine-tuning from v4_m (40 epochs), or about 4.2 hours from COCO (100 epochs). Keep the tab open. `last.pt` is saved to Drive after every epoch, so if Colab disconnects or your GPU time runs out, set `RESUME = True` and run sections 1-5 again later.

In [ ]:
from ultralytics import YOLO

RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)
LAST_WEIGHTS = os.path.join(RUN_DIR, "weights", "last.pt")

if RESUME:
    if not os.path.isfile(LAST_WEIGHTS):
        raise FileNotFoundError(f"No {LAST_WEIGHTS} to resume from. Set RESUME = False.")
    YOLO(LAST_WEIGHTS).train(resume=True)
else:
    if not os.path.isfile(V4_WEIGHTS):
        raise FileNotFoundError(f"v4 weights not found at {V4_WEIGHTS}. Upload final_4cls_ig_v4_m.pt there "
                                "(folder runs/final_4cls_ig_v4_m/weights/, renamed best.pt).")
    model = YOLO(MODEL_VARIANT)
    model.train(
        data=yaml_path,
        **TRAIN_ARGS,
        fliplr=FLIP_LEFT_RIGHT,
        cache="ram",   # same results, faster epochs (Colab's 2 CPUs are the bottleneck)
        seed=0,
        project=RUNS_DIR,
        name=RUN_NAME,
        exist_ok=True,
    )

BEST_WEIGHTS = os.path.join(RUN_DIR, "weights", "best.pt")
print("weights:", BEST_WEIGHTS)


## 6. Scores per class (validation set)

Read each class, not only the mean. A big gap between left and right usually points at labels or direction. The wheeled classes have few validation boxes, so their scores move a lot.

In [ ]:
RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)
BEST_WEIGHTS = os.path.join(RUN_DIR, "weights", "best.pt")

trained = YOLO(BEST_WEIGHTS)
metrics = trained.val(data=yaml_path, split="val", imgsz=IMAGE_SIZE, verbose=False)

print(f"{'class':14} {'boxes':>6} {'P':>6} {'R':>6} {'mAP50':>7} {'mAP50-95':>9}")
for i, c in enumerate(metrics.ap_class_index):
    n_boxes = int(metrics.nt_per_class[int(c)]) if hasattr(metrics, "nt_per_class") else -1
    print(f"{trained.names[int(c)]:14} {n_boxes:6d} {metrics.box.p[i]:6.3f} {metrics.box.r[i]:6.3f} "
          f"{metrics.box.ap50[i]:7.3f} {metrics.box.ap[i]:9.3f}")
print(f"{'all':14} {'':6} {metrics.box.mp:6.3f} {metrics.box.mr:6.3f} {metrics.box.map50:7.3f} {metrics.box.map:9.3f}")

## 7. Curves and sample predictions

- `results.png`: training loss falling while validation loss rises means the model is memorising.
- `confusion_matrix.png`: left/right cells off the diagonal are direction mistakes, person/wheeled cells are riders mistaken for walkers (or the reverse), and the background row is missed objects.

In [ ]:
from IPython.display import Image, display

for name in ("results.png", "confusion_matrix.png", "val_batch0_labels.jpg", "val_batch0_pred.jpg"):
    path = os.path.join(RUN_DIR, name)
    if os.path.exists(path):
        print(name)
        display(Image(filename=path, width=800))

## 8. Try a few validation images

In [ ]:
import random
from PIL import Image as PILImage

val_dir = os.path.join(DATA_DIR, "valid", "images")
random.seed(0)
for name in random.sample(sorted(os.listdir(val_dir)), 3):
    result = trained.predict(os.path.join(val_dir, name), conf=0.40, verbose=False)[0]
    counts = {}
    for c in result.boxes.cls.tolist():
        counts[trained.names[int(c)]] = counts.get(trained.names[int(c)], 0) + 1
    print(name.split(".rf.")[0], counts)
    display(PILImage.fromarray(result.plot()[:, :, ::-1]))

## Next

- Weights: `C389_Supernova_Model/runs/tune_v4_<EXPERIMENT>/weights/best.pt`. Send it (or its Drive path) to whoever runs the comparison. It is compared with v4 26m on the validation set by counting accuracy (exact-count % and count error per photo), not by the mAP above alone.
- Keep the table from section 6 and `results.png` as evidence for the report.
- Only the chosen model is evaluated on test, once. If it is not v4 26m, the report says test was run twice (v4 earlier, then the new model).
- Clear outputs before committing this notebook to git (**Edit → Clear all outputs**).
